# ML Assignment 2 - Classification Models (Training Notebook)
**Name:** Anwesh Sahu  |  **BITS ID:** 2026AK05606

This notebook: loads the dataset, trains 5 classification models, calculates 6 metrics for each, saves the models (`model/` folder), saves `test_data.csv`, and creates `requirements.txt`.

**Run the cells from top to bottom (Shift+Enter on each cell).**

In [ ]:
# CELL 1 - Install / import libraries (safe to run more than once)
#!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib openpyxl   # <-- commented out for smoth run in BITS Lab portal

import os, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")
print("scikit-learn version:", sklearn.__version__)

## Step 1 - Load the dataset
Upload your downloaded dataset file (e.g. `Dry_Bean_Dataset.csv`) to the same folder as this notebook (Colab: click the folder icon on the left, then the upload icon).

Change `DATA_FILE` below if your file name is different. `TARGET_COL` is the column to predict; leave it as `None` to auto-detect (a column named `Class`, otherwise the last column).

In [ ]:
# CELL 2 - Load data
DATA_FILE  = "Dry_Bean_Dataset.csv"   # <-- change if your file name is different (.csv or .xlsx)
TARGET_COL = None                      # <-- None = auto-detect

if DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE)
else:
    df = pd.read_csv(DATA_FILE)

if TARGET_COL is None:
    matches = [c for c in df.columns if c.strip().lower() == "class"]
    TARGET_COL = matches[0] if matches else df.columns[-1]

print("Target column:", TARGET_COL)
print("Rows, Columns:", df.shape)
df.head()

In [ ]:
# CELL 3 - Basic checks (assignment needs >= 12 features and >= 500 rows)
n_rows = df.shape[0]
n_features = df.shape[1] - 1
print("Instances:", n_rows, "| Features:", n_features)
assert n_rows >= 500, "Dataset needs at least 500 instances"
assert n_features >= 12, "Dataset needs at least 12 features"

print("\nMissing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nClass distribution:\n", df[TARGET_COL].value_counts())
df.describe().T

In [ ]:
# CELL 4 - Clean data & make sure all features are numeric
df = df.dropna().reset_index(drop=True)
X = df.drop(columns=[TARGET_COL])
non_numeric = X.select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"These feature columns are not numeric, remove/encode them: {non_numeric}"

le = LabelEncoder()
y = le.fit_transform(df[TARGET_COL])          # text labels -> 0,1,2...
class_names = list(le.classes_)
print("Classes:", class_names)
print("Final shape:", X.shape)

## Step 2 - Train/Test split (80% / 20%)
The 20% test part is saved as `test_data.csv`. This is the file you will upload to the Streamlit app.

In [ ]:
# CELL 5 - Split and save test_data.csv
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

os.makedirs("model", exist_ok=True)
test_df = X_test.copy()
test_df[TARGET_COL] = le.inverse_transform(y_test)   # keep original label names
test_df.to_csv("test_data.csv", index=False)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Saved test_data.csv")

## Step 3 - Define the 5 models and the metric function
Each model is a `Pipeline` (scaling + model) so the saved file already includes preprocessing.

In [ ]:
# CELL 6 - Models + metric function
models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000, random_state=42))]),
    "Decision Tree":       Pipeline([("scaler", StandardScaler()),
                                     ("clf", DecisionTreeClassifier(max_depth=10, random_state=42))]),
    "KNN":                 Pipeline([("scaler", StandardScaler()),
                                     ("clf", KNeighborsClassifier(n_neighbors=7))]),
    "Naive Bayes":         Pipeline([("scaler", StandardScaler()),
                                     ("clf", GaussianNB())]),
    "Support Vector Machine": Pipeline([("scaler", StandardScaler()),
                                     ("clf", SVC(kernel="rbf", probability=True, random_state=42))]),
}

def compute_metrics(model, X_t, y_t):
    y_pred  = model.predict(X_t)
    y_proba = model.predict_proba(X_t)
    n_classes = len(np.unique(y_t))
    if n_classes == 2:
        auc = roc_auc_score(y_t, y_proba[:, 1])
        avg = "binary"
    else:
        auc = roc_auc_score(y_t, y_proba, multi_class="ovr", average="macro")
        avg = "weighted"
    return {
        "Accuracy":  accuracy_score(y_t, y_pred),
        "AUC":       auc,
        "Precision": precision_score(y_t, y_pred, average=avg, zero_division=0),
        "Recall":    recall_score(y_t, y_pred, average=avg, zero_division=0),
        "F1":        f1_score(y_t, y_pred, average=avg, zero_division=0),
        "MCC":       matthews_corrcoef(y_t, y_pred),
    }, y_pred

In [ ]:
# CELL 7 - Train all 5 models and evaluate on the test set
results, predictions = {}, {}
for name, mdl in models.items():
    print("Training:", name, "...")
    mdl.fit(X_train, y_train)
    results[name], predictions[name] = compute_metrics(mdl, X_test, y_test)

results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = "ML Model Name"
results_df

## Step 4 - Save models, metadata and requirements.txt

In [ ]:
# CELL 8 - Save everything
file_names = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib",
}
for name, mdl in models.items():
    joblib.dump(mdl, os.path.join("model", file_names[name]))

joblib.dump({"label_encoder": le, "target_col": TARGET_COL,
             "feature_columns": X.columns.tolist(), "class_names": class_names,
             "model_files": file_names},
            os.path.join("model", "meta.joblib"))
results_df.to_csv("model_results.csv")

with open("requirements.txt", "w") as f:
    f.write(f"streamlit>=1.50\nscikit-learn=={sklearn.__version__}\nnumpy\npandas\nmatplotlib\nseaborn\njoblib\n")
print(open("requirements.txt").read())
print("Saved files:", os.listdir("model"))

## Step 5 - Results for your README

In [ ]:
# CELL 9 - Comparison table in README (markdown) format - copy this output into README.md
print(results_df.reset_index().to_markdown(index=False))
print("\nBest model by MCC:", results_df["MCC"].idxmax())
print("Best model by Accuracy:", results_df["Accuracy"].idxmax())

In [ ]:
# CELL 10 - Chart + confusion matrices (good for screenshots/observations)
results_df.plot(kind="bar", figsize=(11, 5))
plt.title("Model comparison"); plt.ylabel("Score"); plt.ylim(0, 1.05)
plt.xticks(rotation=20); plt.legend(loc="lower right", ncol=3); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (name, pred) in zip(axes.ravel(), predictions.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

best = results_df["MCC"].idxmax()
print("Classification report -", best)
print(classification_report(y_test, predictions[best], target_names=class_names))